<a href="https://colab.research.google.com/github/hammad-5992/Hammad_23122005_Lab1_AI_6A/blob/main/Hammad_Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install once in Google Colab
!pip install -U google-genai

from google import genai
import logging
logging.getLogger("google_genai.models").setLevel(logging.ERROR)
client = genai.Client(api_key="new_key")

MODEL = "gemini-3.6-flash"


# =====================================================
# UNIVERSITY DATA
# =====================================================

university_name = "Federal Urdu University of Arts, Science & Technology"
campus = "Gulshan-e-Iqbal, Karachi"
department = "Computer Science"
hod_name = "Dr. Siddiq"

program = "BSCS"
semester = "6th"
section = "A"

# List of courses (each course is its own dictionary)
courses = [
    {"code": "AI", "name": "Artificial Intelligence", "teacher": "Ms. Uzma"},
    {"code": "NLP", "name": "Natural Language Processing", "teacher": "Dr. Khalid"},
    {"code": "NSC", "name": "Numerical and Symbolic Computing", "teacher": "Dr. Akhtar Raza"},
    {"code": "CC", "name": "Compiler Construction", "teacher": "Ms. Naheed"},
    {"code": "PP", "name": "Professional Practices", "teacher": "Not mentioned"},
]

# Timetable organized as a simple list of classes per day
timetable = {
    "Monday": [
        "12:20 - 1:10  : NSC - Lab 1",
        "2:00 - 2:50   : NLP - Lab 1",
        "2:50 - 3:40   : AI - Lab 1",
    ],
    "Tuesday": [
        "12:20 - 1:10  : CC - Lab 1",
        "2:00 - 2:50   : AI - Lab 1",
        "2:50 - 3:40   : NSC - Lab 1",
        "3:40 - 4:30   : NLP",
    ],
    "Wednesday": [
        "12:20 - 1:10  : PP - Room 2",
        "2:00 - 2:50   : NSC - Room 2",
        "2:50 - 3:40   : AI - Lab 1",
    ],
    "Thursday": [
        "10:40 - 11:30 : NSC - Lab 1",
        "11:30 - 12:20 : CC - Lab 1",
        "12:20 - 1:10  : PP - Lab 1",
        "2:00 - 2:50   : NLP - Lab 1",
        "2:50 - 3:40   : AI - Lab 1",
    ],
    "Friday": [
        "11:30 - 12:20 : CC - Lab 2",
        "12:20 - 1:10  : PP - Room 1",
    ],
}


# =====================================================
# SIMPLE HELPER FUNCTIONS
# =====================================================

def find_teacher(course_code):
    """Looks through the courses list and returns the teacher's name."""
    for course in courses:
        if course["code"].lower() == course_code.lower():
            return course["teacher"]
    return "Course not found."


def build_info_text():
    """Turns all the data above into one text block for the AI prompt."""
    text = f"University: {university_name}\n"
    text += f"Campus: {campus}\n"
    text += f"Department: {department}, HOD: {hod_name}\n"
    text += f"Program: {program}, Semester: {semester}, Section: {section}\n\n"

    text += "Courses:\n"
    for course in courses:
        text += f"  - {course['code']} ({course['name']}) — Teacher: {course['teacher']}\n"

    text += "\nTimetable:\n"
    for day, classes in timetable.items():
        text += f"  {day}:\n"
        for c in classes:
            text += f"    {c}\n"

    return text


# =====================================================
# ROLE-BASED AGENT
# =====================================================

def department_agent(user_input):
    info = build_info_text()

    prompt = f"""
You are FUUAST CS Assistant, a helpful AI agent for the BSCS 6th Semester,
Section A class at Federal Urdu University of Arts, Science & Technology.

Use this information whenever the question is about courses, teachers,
timings, or rooms:

{info}

For anything else — general questions or other topics — answer normally
using your own knowledge. Keep answers short and simple.

Question: {user_input}
Answer:
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )
    return response.text


# =====================================================
# CHAT LOOP
# =====================================================

print("FUUAST CS Department Agent")
print("BSCS - 6th Semester - Section A")
print("Type 'teacher AI' to quickly check who teaches a course, or 'exit' to quit\n")

while True:
    user = input("You: ")

    if user.lower() == "exit":
        print("Goodbye!")
        break

    # Quick lookup feature — doesn't need to call the AI at all
    if user.lower().startswith("teacher"):
        parts = user.split(maxsplit=1)
        if len(parts) == 2:
            teacher = find_teacher(parts[1].strip())
            print(f"\nAgent: {teacher}\n")
        else:
            print("Please type it like: teacher AI\n")
        continue

    try:
        reply = department_agent(user)
        print("\nAgent:", reply)
        print()
    except Exception as e:
        print("Error:", e)

FUUAST CS Department Agent
BSCS - 6th Semester - Section A
Type 'teacher AI' to quickly check who teaches a course, or 'exit' to quit

You: What courses do I have on Monday?



Agent: On Monday, you have the following classes:

- **12:20 - 1:10**: NSC (Numerical and Symbolic Computing) — Lab 1
- **2:00 - 2:50**: NLP (Natural Language Processing) — Lab 1
- **2:50 - 3:40**: AI (Artificial Intelligence) — Lab 1

You: Who teaches Natural Language Processing?

Agent: Dr. Khalid teaches Natural Language Processing (NLP).

You: Who is the HOD of the Computer Science department?

Agent: The HOD of the Computer Science department is **Dr. Siddiq**.

You: teacher AI

Agent: Ms. Uzma

You: How many labs do I have this week?

Agent: You have **13 lab sessions** scheduled this week:

* **Monday:** 3 labs (NSC, NLP, AI)
* **Tuesday:** 3 labs (CC, AI, NSC)
* **Wednesday:** 1 lab (AI)
* **Thursday:** 5 labs (NSC, CC, PP, NLP, AI)
* **Friday:** 1 lab (CC)

You: exit
Goodbye!
